# 01 · Go rules and an independent verifier

This notebook establishes the rule boundary used by the AHR Go experiments.
**The project default is 9×9 with 7.5 komi.** The same implementation supports
3×3 and 5×5 for controlled checks. Validation results appear at the end.

**Inputs:** Python 3.12+ and this notebook; no game dataset, engine, GPU, or
project source file is required. `nbformat` is used to record source provenance.
**Outputs:** `reports/01_rules_validation.json` and saved cell outputs.
**Expected runtime:** under two minutes on an ordinary CPU; no training occurs.
**Resume:** there is no long-running stage or resumable training state. If a
cell is interrupted, restart the kernel and choose **Run All**. The report is
replaced atomically only after the checks complete; an earlier report records
its own timestamp and notebook-code hash.

Export cells carry the tags `ahr-export` and `rules`. Later notebooks execute
only those definition cells; importing the API does not run tests or write reports.

## Rules and limits

Black is `1`, White is `-1`, and empty is `0`. A move is a row-major integer;
`size * size` is pass. Captures occur before checking suicide. Repeating any
earlier board is illegal under positional superko; passes are exempt. Two
consecutive passes end the game. Black's score is Black area minus White area
minus komi. Area includes stones and empty regions bordered by exactly one
color. Mixed-border and borderless regions are neutral.

Players must capture dead stones before passing. There is no automatic
dead-stone removal, resignation rule, or move-cap adjudication in this API.
A nonterminal area score is a board evaluation, not an optimal-play value.
The checks below provide evidence of correctness, **not a formal proof** of
the rule implementation or a proof of strong Go play.

## 1. Locate the project and set bounded checks

Run from `ahr_go` or its `notebooks` directory. Local staging is also supported.
Set `AHR_GO_PROJECT` only when working from a different directory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import hashlib
import json
import os
import random
import time
import nbformat

current_directory = Path.cwd().resolve()
if os.environ.get("AHR_GO_PROJECT") or os.environ.get("AHR_GO_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ.get("AHR_GO_PROJECT") or os.environ["AHR_GO_PROJECT_ROOT"]).expanduser().resolve()
elif current_directory.name == "notebooks":
    PROJECT_ROOT = current_directory.parent
elif (current_directory / "notebooks").is_dir():
    PROJECT_ROOT = current_directory
elif (current_directory / "remote_notebooks/ahr_go/notebooks").is_dir():
    PROJECT_ROOT = current_directory / "remote_notebooks/ahr_go"
else:
    raise RuntimeError("Run from ahr_go or ahr_go/notebooks, or set AHR_GO_PROJECT_ROOT.")
REPORT_PATH = PROJECT_ROOT / "reports/01_rules_validation.json"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/01_go_rules_and_independent_verifier.ipynb"
VALIDATION_SEED = 20260923
SEEDS_PER_SIZE = 6
MAX_REPLAY_MOVES = 50
print(f"Project: {PROJECT_ROOT}")
print("Bounded CPU checks: 3×3, 5×5, 9×9; six deterministic replay seeds per size.")

## 2. Export the immutable board state

`GoState` includes the complete positional-superko history. Public construction
and JSON deserialization validate board dimensions, strict integer types, and
finite komi. Boolean and floating-point stones/counters are rejected. Every exported
cell contains only imports, constants, classes, and functions.

In [ ]:
from dataclasses import dataclass
from functools import lru_cache
from math import isfinite
from typing import Any, Iterable, Mapping


BLACK, WHITE, EMPTY = 1, -1, 0
RULESET = {
    "name": "ahr_go_area_positional_superko_v1",
    "default_size": 9,
    "default_komi": 7.5,
    "suicide": False,
    "ko": "positional_superko",
    "passes_exempt_from_superko": True,
    "consecutive_passes_to_end": 2,
    "scoring": "stones_plus_single_color_bordered_empty_regions_minus_komi",
    "dead_stones": "must_be_captured_before_passing",
    "move_cap": None,
}
RULES_ORIGIN_SHA256 = "9e715328359f56c2003071a327328b618c20999ad54b058026568eba141b81d6"


class IllegalMove(ValueError):
    """The requested action violates the configured Go rules."""


@lru_cache(maxsize=16)
def _neighbors(size: int) -> tuple[tuple[int, ...], ...]:
    return tuple(
        tuple(
            rr * size + cc
            for rr, cc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1))
            if 0 <= rr < size and 0 <= cc < size
        )
        for r in range(size)
        for c in range(size)
    )


@dataclass(frozen=True, slots=True)
class GoState:
    size: int
    board: tuple[int, ...]
    to_play: int = 1
    komi: float = 7.5
    history: frozenset[tuple[int, ...]] = frozenset()
    consecutive_passes: int = 0
    move_number: int = 0

    def __post_init__(self) -> None:
        if type(self.size) is not int or self.size < 2:
            raise ValueError("Board size must be an integer of at least 2; bool is not accepted.")
        board = tuple(self.board)
        if len(board) != self.size * self.size or any(type(x) is not int or x not in (-1, 0, 1) for x in board):
            raise ValueError("Board must contain size*size integer entries from {-1, 0, 1}; bool is not accepted.")
        if type(self.to_play) is not int or self.to_play not in (-1, 1):
            raise ValueError("to_play must be integer Black (1) or White (-1); bool is not accepted.")
        if type(self.komi) not in (int, float) or not isfinite(self.komi):
            raise ValueError("komi must be a finite real number; bool is not accepted.")
        if (type(self.consecutive_passes) is not int or self.consecutive_passes not in (0, 1, 2)
                or type(self.move_number) is not int or self.move_number < 0):
            raise ValueError("Pass count and move number must be valid integers; bool is not accepted.")
        # Validate before deduplication: 0.0 and False compare equal to integer 0.
        history_boards = tuple(tuple(item) for item in self.history)
        if any(len(item) != len(board) or any(type(x) is not int or x not in (-1, 0, 1) for x in item)
               for item in history_boards):
            raise ValueError("History contains an invalid board or non-integer stone.")
        object.__setattr__(self, "board", board)
        object.__setattr__(self, "history", frozenset(history_boards) | {board})


    @property
    def terminal(self) -> bool:
        return self.consecutive_passes == 2

    @property
    def pass_action(self) -> int:
        return self.size * self.size

    def to_dict(self) -> dict[str, Any]:
        return to_dict(self)

    @classmethod
    def from_dict(cls, value: Mapping[str, Any]) -> "GoState":
        return from_dict(value)

### Groups and liberties

`initial_state(size=9, komi=7.5)` creates an empty board. Group helpers expose
connected stones and their empty neighbors for tactical constraints downstream.

In [ ]:
def initial_state(size: int = 9, komi: float = 7.5) -> GoState:
    return GoState(size=size, board=(0,) * (size * size), komi=komi)


def group_and_liberties(
    board: Iterable[int], size: int, point: int
) -> tuple[frozenset[int], frozenset[int]]:
    """Return the connected stone group and its empty neighboring points."""
    values = tuple(board)
    if not 0 <= point < size * size or not values[point]:
        raise ValueError("point must contain a stone.")
    color = values[point]
    group = {point}
    liberties: set[int] = set()
    pending = [point]
    adjacent = _neighbors(size)
    while pending:
        current = pending.pop()
        for neighbor in adjacent[current]:
            if values[neighbor] == 0:
                liberties.add(neighbor)
            elif values[neighbor] == color and neighbor not in group:
                group.add(neighbor)
                pending.append(neighbor)
    return frozenset(group), frozenset(liberties)


def groups_and_liberties(
    board: Iterable[int], size: int
) -> tuple[tuple[frozenset[int], frozenset[int]], ...]:
    """Return each stone group exactly once, in board-point order."""
    values = tuple(board)
    visited: set[int] = set()
    result = []
    for point, color in enumerate(values):
        if color and point not in visited:
            group, liberties = group_and_liberties(values, size, point)
            visited.update(group)
            result.append((group, liberties))
    return tuple(result)

### Legal transitions

`play(state, action)` returns a new state or raises `IllegalMove`.
The private successor constructor reuses already-validated history; all move
checks still run. `legal_actions(state)` includes pass unless the game ended.

In [ ]:
def _trusted_successor(
    state: GoState,
    board: tuple[int, ...],
    history: frozenset[tuple[int, ...]],
    consecutive_passes: int,
) -> GoState:
    """Construct a successor after ``play`` has checked its rule invariants.

    Public construction and deserialization validate all supplied history.
    Internally, every earlier board is already validated and the new board is
    created by ``play``; rescanning the full history at every search node adds
    work proportional to the game's length without adding a new check.
    """
    child = object.__new__(GoState)
    for name, value in (
        ("size", state.size),
        ("board", board),
        ("to_play", -state.to_play),
        ("komi", state.komi),
        ("history", history),
        ("consecutive_passes", consecutive_passes),
        ("move_number", state.move_number + 1),
    ):
        object.__setattr__(child, name, value)
    return child


def play(state: GoState, action: int) -> GoState:
    if state.terminal:
        raise IllegalMove("The game has already ended after two passes.")
    if type(action) is not int or not 0 <= action <= state.pass_action:
        raise IllegalMove("Action is outside the board and is not pass.")
    if action == state.pass_action:
        return _trusted_successor(
            state, state.board, state.history, state.consecutive_passes + 1,
        )
    if state.board[action] != 0:
        raise IllegalMove("The point is occupied.")
    board = list(state.board)
    board[action] = state.to_play
    checked: set[int] = set()
    for neighbor in _neighbors(state.size)[action]:
        if board[neighbor] == -state.to_play and neighbor not in checked:
            group, liberties = group_and_liberties(board, state.size, neighbor)
            checked.update(group)
            if not liberties:
                for stone in group:
                    board[stone] = 0
    _, own_liberties = group_and_liberties(board, state.size, action)
    if not own_liberties:
        raise IllegalMove("Suicide is not permitted.")
    new_board = tuple(board)
    if new_board in state.history:
        raise IllegalMove("Positional superko forbids repeating an earlier board.")
    return _trusted_successor(state, new_board, state.history | {new_board}, 0)


def legal_actions(state: GoState) -> tuple[int, ...]:
    if state.terminal:
        return ()
    legal = []
    for action in range(state.pass_action):
        if state.board[action]:
            continue
        try:
            play(state, action)
        except IllegalMove:
            continue
        legal.append(action)
    return tuple(legal) + (state.pass_action,)

### Area score and complete serialization

`area_score` is always from Black's perspective. JSON conversion preserves
board history, side to play, komi, pass count, and move number.

In [ ]:
def area_score(state: GoState) -> float:
    """Black area minus White area minus komi, also usable before game end.

    A nonterminal score is only a board evaluation, not a proven game result.
    Empty regions bordered by both colors, or neither color, count for neither.
    """
    score = float(sum(state.board)) - state.komi
    visited: set[int] = set()
    adjacent = _neighbors(state.size)
    for point, color in enumerate(state.board):
        if color or point in visited:
            continue
        region = {point}
        visited.add(point)
        pending = [point]
        borders: set[int] = set()
        while pending:
            current = pending.pop()
            for neighbor in adjacent[current]:
                if state.board[neighbor]:
                    borders.add(state.board[neighbor])
                elif neighbor not in visited:
                    visited.add(neighbor)
                    region.add(neighbor)
                    pending.append(neighbor)
        if len(borders) == 1:
            score += borders.pop() * len(region)
    return score


def is_terminal(state: GoState) -> bool:
    return state.terminal


def to_dict(state: GoState) -> dict[str, Any]:
    """JSON-compatible complete state, including superko history."""
    return {
        "size": state.size,
        "board": list(state.board),
        "to_play": state.to_play,
        "komi": state.komi,
        "history": [list(board) for board in sorted(state.history)],
        "consecutive_passes": state.consecutive_passes,
        "move_number": state.move_number,
    }


def from_dict(value: Mapping[str, Any]) -> GoState:
    return GoState(
        size=value["size"],
        board=tuple(value["board"]),
        to_play=value["to_play"],
        komi=value["komi"],
        history=tuple(tuple(board) for board in value["history"]),
        consecutive_passes=value["consecutive_passes"],
        move_number=value["move_number"],
    )

## 3. Define a second implementation

This verifier uses coordinate maps and independently finds captured groups.
Its scorer recomputes territory separately from every empty point. Neither
function calls the production group, move, or scoring helpers. These are
validation functions, so they are deliberately excluded from exported cells.

In [ ]:
def reference_move(state, action):
    """Independent coordinate-map implementation; return None if illegal."""
    n = state.size
    if state.consecutive_passes == 2 or action < 0 or action > n * n:
        return None
    if action == n * n:
        return state.board
    grid = {(r, c): state.board[r * n + c] for r in range(n) for c in range(n)}
    location = divmod(action, n)
    if grid[location]:
        return None
    grid[location] = state.to_play

    def adjacent(rc):
        r, c = rc
        return [(x, y) for x, y in ((r, c - 1), (r, c + 1), (r - 1, c), (r + 1, c)) if (x, y) in grid]

    def component(origin):
        color = grid[origin]
        found = {origin}
        frontier = [origin]
        liberty = False
        for point in frontier:
            for other in adjacent(point):
                if grid[other] == 0:
                    liberty = True
                if grid[other] == color and other not in found:
                    found.add(other)
                    frontier.append(other)
        return found, liberty

    # Determine all opposing dead groups before removing any, independently of
    # the production implementation's neighbor-by-neighbor removal order.
    captured = set()
    for other in adjacent(location):
        if grid[other] == -state.to_play:
            stones, has_liberty = component(other)
            if not has_liberty:
                captured |= stones
    for stone in captured:
        grid[stone] = 0
    _, has_liberty = component(location)
    if not has_liberty:
        return None
    result = tuple(grid[(r, c)] for r in range(n) for c in range(n))
    return None if result in state.history else result


def reference_score(state):
    """Independent per-empty-point reachability scoring (no shared helpers)."""
    n = state.size
    total = -state.komi
    for start, occupant in enumerate(state.board):
        if occupant:
            total += occupant
            continue
        discovered = {start}
        frontier = [start]
        sees_black = False
        sees_white = False
        for current in frontier:
            row, col = divmod(current, n)
            for rr, cc in ((row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)):
                if not (0 <= rr < n and 0 <= cc < n):
                    continue
                other = rr * n + cc
                color = state.board[other]
                sees_black |= color == 1
                sees_white |= color == -1
                if color == 0 and other not in discovered:
                    discovered.add(other)
                    frontier.append(other)
        if sees_black and not sees_white:
            total += 1
        if sees_white and not sees_black:
            total -= 1
    return float(total)

## 4. Check fixtures, all 3×3 static scores, and legal replays

Coverage includes single and multiple captures, suicide, a ko recapture,
non-immediate superko history, pass termination, scoring and serialization.
All `3^9 = 19,683` static 3×3 layouts are checked for scoring agreement.
All 2×2 layouts and both sides to play are checked for transition agreement.
These exhaustive layout checks do not claim that every layout is reachable.

Random legal replays then exercise 3×3, 5×5, and 9×9 histories. At selected
replay points, all eight board symmetries must preserve scores, legal moves,
and transformed successor states, including transformed historical boards.

In [ ]:
validation_counts = {
    "fixture_assertions": 0,
    "malformed_state_rejections": 0,
    "exhaustive_3x3_score_layouts": 0,
    "exhaustive_2x2_position_turn_pairs": 0,
    "move_comparisons": 0,
    "replay_positions_by_size": {"3": 0, "5": 0, "9": 0},
    "serialization_roundtrips": 0,
    "symmetry_state_checks": 0,
    "symmetry_transition_checks": 0,
}

def require(condition, message):
    if not condition:
        raise AssertionError(message)

def fixture(condition, message):
    require(condition, message)
    validation_counts["fixture_assertions"] += 1

def requires_illegal(state, action, expected_reason):
    try:
        play(state, action)
    except IllegalMove as error:
        fixture(expected_reason.lower() in str(error).lower(), str(error))
    else:
        raise AssertionError(f"Expected {expected_reason} for action {action}")

def requires_invalid_state(payload):
    # Round-trip through JSON so these tests cover deserialized external data.
    # Python's JSON decoder accepts NaN/Infinity; the state boundary rejects them.
    decoded = json.loads(json.dumps(payload))
    try:
        from_dict(decoded)
    except (ValueError, TypeError, KeyError):
        validation_counts["malformed_state_rejections"] += 1
    else:
        raise AssertionError("Malformed state was accepted by from_dict")

def compare_moves(state):
    reference_legal = []
    for action in range(state.pass_action + 1):
        expected = reference_move(state, action)
        try:
            actual = play(state, action).board
        except IllegalMove:
            actual = None
        require(actual == expected, f"Move disagreement: size={state.size}, action={action}")
        if expected is not None:
            reference_legal.append(action)
        validation_counts["move_comparisons"] += 1
    require(tuple(reference_legal) == legal_actions(state), "Legal-action set disagreement")

def transformed_point(point, size, rotation, reflection):
    if point == size * size:
        return point
    row, col = divmod(point, size)
    if reflection:
        col = size - 1 - col
    for _ in range(rotation):
        row, col = col, size - 1 - row
    return row * size + col

def transformed_state(state, rotation, reflection):
    def transform_board(board):
        result = [0] * len(board)
        for point, color in enumerate(board):
            result[transformed_point(point, state.size, rotation, reflection)] = color
        return tuple(result)
    return GoState(
        size=state.size, board=transform_board(state.board), to_play=state.to_play,
        komi=state.komi, history=frozenset(transform_board(b) for b in state.history),
        consecutive_passes=state.consecutive_passes, move_number=state.move_number,
    )

def check_symmetries(state):
    original_actions = legal_actions(state)
    for reflection in (False, True):
        for rotation in range(4):
            transformed = transformed_state(state, rotation, reflection)
            expected_actions = {transformed_point(a, state.size, rotation, reflection)
                                for a in original_actions}
            require(set(legal_actions(transformed)) == expected_actions, "Symmetry legality failed")
            require(area_score(transformed) == area_score(state), "Symmetry score failed")
            validation_counts["symmetry_state_checks"] += 1
            for action in original_actions:
                mapped = transformed_point(action, state.size, rotation, reflection)
                expected_child = transformed_state(play(state, action), rotation, reflection)
                require(play(transformed, mapped) == expected_child, "Symmetry transition failed")
                validation_counts["symmetry_transition_checks"] += 1

def run_validation():
    started = time.perf_counter()
    default = initial_state()
    fixture((default.size, default.komi, default.to_play) == (9, 7.5, BLACK), "Default rules changed")
    fixture(len(legal_actions(default)) == 82, "Empty 9x9 must offer 81 points and pass")
    fixture(default.history == frozenset({(0,) * 81}), "Initial history missing")
    occupied = play(default, 40)
    requires_illegal(occupied, 40, "occupied")
    for action in (-1, 82, 1.5, True, False):
        requires_illegal(default, action, "outside")
    fixture(default.board == (0,) * 81, "play mutated its input")
    for field, bad_value in (
        ("size", 9.0), ("size", True),
        ("to_play", 1.0), ("to_play", True),
        ("consecutive_passes", 0.0), ("consecutive_passes", False),
        ("move_number", 0.0), ("move_number", False), ("move_number", -1),
        ("komi", float("nan")), ("komi", float("inf")), ("komi", -float("inf")),
        ("komi", True), ("komi", "7.5"),
    ):
        payload = to_dict(default)
        payload[field] = bad_value
        requires_invalid_state(payload)
    for bad_stone in (0.0, False, 1.0, True):
        payload = to_dict(default)
        payload["board"][0] = bad_stone
        requires_invalid_state(payload)
        payload = to_dict(default)
        malformed_history = list(default.board)
        malformed_history[0] = bad_stone
        payload["history"].append(malformed_history)
        requires_invalid_state(payload)
    incomplete_payload = to_dict(default)
    del incomplete_payload["history"]
    requires_invalid_state(incomplete_payload)

    single = GoState(3, (0, -1, 1, -1, 1, 0, 0, 0, 0))
    fixture(play(single, 0).board == (1, 0, 1, -1, 1, 0, 0, 0, 0), "Single capture failed")
    connected = GoState(3, (-1, -1, 1, 1, 0, 0, 0, 0, 0))
    fixture(play(connected, 4).board == (0, 0, 1, 1, 1, 0, 0, 0, 0), "Connected-group capture failed")
    multiple = GoState(3, (1, -1, 1, -1, 0, -1, 1, -1, 1))
    fixture(play(multiple, 4).board == (1, 0, 1, 0, 1, 0, 1, 0, 1), "Multiple captures failed")
    for board in ((0, -1, 0, -1, 0, -1, 0, -1, 0),
                  (-1, 1, -1, -1, 0, -1, 0, -1, 0)):
        requires_illegal(GoState(3, board), 4, "suicide")

    ko_board = [0] * 25
    for row, col in ((0, 1), (1, 0), (2, 1)):
        ko_board[row * 5 + col] = BLACK
    for row, col in ((1, 1), (0, 2), (1, 3), (2, 2)):
        ko_board[row * 5 + col] = WHITE
    ko_start = GoState(5, tuple(ko_board))
    ko_capture = play(ko_start, 7)
    requires_illegal(ko_capture, 6, "superko")
    # A synthetic history-membership fixture targets a repeat without a
    # current ko shape; it does not assert reachability of this history.
    old_child = play(initial_state(3), 0).board
    historical = GoState(3, (0,) * 9, history=frozenset({old_child, (0,) * 9}))
    requires_illegal(historical, 0, "superko")
    restored_ko = from_dict(json.loads(json.dumps(to_dict(ko_capture))))
    fixture(restored_ko == ko_capture, "Full-state serialization failed")
    requires_illegal(restored_ko, 6, "superko")

    passed = play(default, default.pass_action)
    ended = play(passed, passed.pass_action)
    fixture(passed.history == default.history, "Pass changed superko history")
    fixture(ended.terminal and legal_actions(ended) == (), "Two passes did not end game")
    requires_illegal(ended, 0, "ended")
    fixture(play(passed, 0).consecutive_passes == 0, "Stone move did not reset pass count")
    fixture(area_score(default) == -7.5, "Empty board score incorrect")
    fixture(area_score(GoState(3, (1, 1, 1, 1, 0, 1, 1, 1, 1))) == 1.5, "Black territory incorrect")
    fixture(area_score(GoState(3, (-1, -1, -1, -1, 0, -1, -1, -1, -1))) == -16.5, "White territory incorrect")
    fixture(area_score(GoState(3, (1, 0, -1, 1, 0, -1, 1, 0, -1))) == -7.5, "Neutral territory incorrect")
    group, liberties = group_and_liberties((1, 1, 0, 1, -1, 0, 0, 0, -1), 3, 0)
    fixture(group == frozenset({0, 1, 3}) and liberties == frozenset({2, 6}), "Group helper incorrect")

    for layout in product((-1, 0, 1), repeat=9):
        state = GoState(3, layout)
        require(area_score(state) == reference_score(state), "Exhaustive static-score disagreement")
        validation_counts["exhaustive_3x3_score_layouts"] += 1
    for layout in product((-1, 0, 1), repeat=4):
        for color in (BLACK, WHITE):
            compare_moves(GoState(2, layout, to_play=color))
            validation_counts["exhaustive_2x2_position_turn_pairs"] += 1

    for size in (3, 5, 9):
        for seed in range(SEEDS_PER_SIZE):
            rng = random.Random(VALIDATION_SEED + 1000 * size + seed)
            state = initial_state(size)
            for step in range(MAX_REPLAY_MOVES):
                compare_moves(state)
                require(area_score(state) == reference_score(state), "Replay-score disagreement")
                require(from_dict(json.loads(json.dumps(to_dict(state)))) == state, "Replay serialization failed")
                validation_counts["serialization_roundtrips"] += 1
                validation_counts["replay_positions_by_size"][str(size)] += 1
                if step % 15 == 0:
                    check_symmetries(state)
                if state.terminal:
                    break
                nonpasses = [a for a in legal_actions(state) if a != state.pass_action]
                action = rng.choice(nonpasses) if nonpasses and rng.random() >= .08 else state.pass_action
                state = play(state, action)
    return time.perf_counter() - started

validation_elapsed_seconds = run_validation()
print(f"Passed all rule checks in {validation_elapsed_seconds:.2f} seconds.")

## 5. Save the validation report

Counts describe the checks actually executed. The report hashes the notebook's
code cells so downstream work can identify the tested rule version. The
independent verifier shares the declared rule specification, but it does not
share the production move or scoring implementation.

In [ ]:
saved_notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
nbformat.validate(saved_notebook)
code_payload = "\n\n".join(cell.source for cell in saved_notebook.cells if cell.cell_type == "code")
export_payload = "\n\n".join(cell.source for cell in saved_notebook.cells
                                  if "ahr-export" in cell.metadata.get("tags", []))
validation_report = {
    "status": "passed",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "ruleset": RULESET,
    "seed": VALIDATION_SEED,
    "counts": validation_counts,
    "elapsed_seconds": validation_elapsed_seconds,
    "notebook_code_sha256": hashlib.sha256(code_payload.encode()).hexdigest(),
    "exported_rules_sha256": hashlib.sha256(export_payload.encode()).hexdigest(),
    "origin_local_rules_sha256": RULES_ORIGIN_SHA256,
    "independent_implementation": "coordinate-map move oracle and per-empty-point score oracle",
    "formal_proof": False,
    "training_performed": False,
    "limitations": [
        "Exhaustive static layout checks are not exhaustive game-history checks.",
        "Random 9x9 replays are sampled coverage.",
        "Players must capture dead stones before passing.",
        "No claim about Go playing strength or optimal moves is established.",
    ],
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_report = REPORT_PATH.with_suffix(".json.tmp")
temporary_report.write_text(json.dumps(validation_report, indent=2) + "\n")
os.replace(temporary_report, REPORT_PATH)
print("Rules verifier: PASSED")
print(f"Static 3×3 layouts scored: {validation_counts['exhaustive_3x3_score_layouts']:,}")
print(f"Move comparisons: {validation_counts['move_comparisons']:,}")
print(f"Replay positions by board size: {validation_counts['replay_positions_by_size']}")
print(f"Symmetry transitions: {validation_counts['symmetry_transition_checks']:,}")
print(f"Saved: {REPORT_PATH}")

## Interpretation and next step

A successful run establishes agreement across the reported fixtures, exhaustive
static checks, and sampled replays. It is a tested implementation of the stated
rules. It does not establish game-theoretic correctness for every possible
history, strong play, or an advantage from AHR.

The following notebooks may import the tagged rule definitions to construct
Go constraint problems and verified training examples. Preserve full histories
when storing states: board stones alone are insufficient for positional-superko
legality. Keep terminal results separate from finite-horizon evaluations.